In [1]:
import JSON
using JuMP
using Gurobi

current_directory = @__DIR__

filename = "DE_L_results_T_10_delta_5_scen_35_trial_1_inv_1_cap._1_cap.inc._1.json"
relative_path = joinpath(current_directory, filename)
# println(relative_path)
data = JSON.parsefile(relative_path)


Dict{String, Any} with 9 entries:
  "Y"  => Dict{String, Any}("Sanofi"=>Dict{String, Any}("3"=>1.0, "4"=>1.0, "1"…
  "Q"  => Dict{String, Any}("PCV"=>Dict{String, Any}("Serum_Institute"=>Dict{St…
  "I"  => Dict{String, Any}("PCV"=>Dict{String, Any}("4"=>Dict{String, Any}("24…
  "W"  => Dict{String, Any}("Sanofi"=>Dict{String, Any}("3"=>Dict{String, Any}(…
  "X"  => Dict{String, Any}("PCV"=>Dict{String, Any}("Serum_Institute"=>Dict{St…
  "S"  => Dict{String, Any}("PCV"=>Dict{String, Any}("4"=>Dict{String, Any}("24…
  "L"  => Dict{String, Any}("Sanofi"=>Dict{String, Any}("3"=>1.0, "4"=>1.0, "1"…
  "Vc" => Dict{String, Any}("PCV"=>Dict{String, Any}("3"=>Dict{String, Any}("24…
  "F"  => Dict{String, Any}("PCV"=>Dict{String, Any}("3"=>Dict{String, Any}("4"…

In [2]:
for (p, time) in data["Y"]
    for (t, val) in time
        value = get(data["Y"], p, 0)[t]  # Safely get the value or a default Dict
        # println(value)
        if value != 0
            # println("Exists")
        else
            println("No key for: $p at time $t")
        end
    end
end

No key for: Merck_Sharp at time 1
No key for: Merck_Sharp at time 5
No key for: Merck_Sharp at time 2
No key for: Merck_Sharp at time 10
No key for: Merck_Sharp at time 9


In [3]:
P = ["AJ_Vaccines","BB_NCIPD","China_National","Bharat_Biotech","Bilthoven","Biological_E","GSK","Haffkine_Bio",
"LG_Chem","Merck_Sharp","Panacea_Biotec","PT_Bio","Sanofi","Serum_Institute","Pfizer"]

A = ["Measles", "Mumps", "Rubella", "Diphtheria", "Tetanus", "Pertussis", "Hepatitis_B", "Hib", "Polio", "HPV", "Rotavirus", "PCV"]

V = ["M", "MR", "MMR", "TT", "HepB", "Hib", "IPV", "OPV", "DT", "Td", "DTwP", "DTwP-Hib", "Penta", "Hexa", "HPV", "Rotavirus", "PCV"]

P_v = Dict("M" => ["Serum_Institute", "PT_Bio"], "MR" => ["Serum_Institute", "Biological_E"], "MMR" => ["Serum_Institute","GSK"],
"TT"=> ["Serum_Institute","PT_Bio","BB_NCIPD", "Biological_E"], "HepB" => ["Serum_Institute","LG_Chem"], "Hib" => ["Serum_Institute"],
"IPV" => ["LG_Chem","AJ_Vaccines","Bilthoven","Sanofi"],
"OPV" => ["Serum_Institute","PT_Bio","GSK","Sanofi","Panacea_Biotec","China_National","Bharat_Biotech","Haffkine_Bio"],
"DT" => ["PT_Bio","BB_NCIPD"], "Td" => ["Serum_Institute","PT_Bio","BB_NCIPD", "Biological_E"], "DTwP" => ["Serum_Institute","Biological_E"], "DTwP-Hib" => ["Serum_Institute"],
"Penta" => ["Serum_Institute","PT_Bio","Biological_E","LG_Chem","Panacea_Biotec"], "Hexa" => ["Sanofi"],
"HPV" => ["GSK","Merck_Sharp","China_National"], "Rotavirus" => ["Serum_Institute","GSK","Bharat_Biotech"], "PCV" => ["Serum_Institute","GSK","Pfizer"])

max_tender_length = 5
tmin = 1
tmax = 10
T = [t for t in tmin:tmax]
T_initial = [t for t in tmin-1:tmax]
Δ = [i for i in 1:max_tender_length]

F_time_set = []
for t in T
    for tau in T
        if tau >= t
            if (tau - t + 1) in Δ
                push!(F_time_set, (t, tau))
            end
        end
    end
end

In [4]:
gurobi_solver_DE = JuMP.optimizer_with_attributes(Gurobi.Optimizer, "FeasibilityTol" => 1e-4, "OutputFlag" => 1, "Presolve" => 1, 
"NumericFocus" => 1, "MIPGap" => 1e-2, "Threads" => 8) 
model = Model(gurobi_solver_DE)
@variable(model, Y[p in P, t in T], Bin)
@variable(model, Q[v in V, p in P_v[v], (t, tau) in F_time_set] >= 0)
@variable(model, W[p in P, (t, tau) in F_time_set], Bin)
@variable(model, F[a in A, (t, tau) in F_time_set], Bin)
@variable(model, L[p in P, t in T] >= 0)

Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-04
Set parameter FeasibilityTol to value 0.0001
Set parameter Presolve to value 1
Set parameter NumericFocus to value 1
Set parameter MIPGap to value 0.01
Set parameter Threads to value 8


2-dimensional DenseAxisArray{VariableRef,2,...} with index sets:
    Dimension 1, ["AJ_Vaccines", "BB_NCIPD", "China_National", "Bharat_Biotech", "Bilthoven", "Biological_E", "GSK", "Haffkine_Bio", "LG_Chem", "Merck_Sharp", "Panacea_Biotec", "PT_Bio", "Sanofi", "Serum_Institute", "Pfizer"]
    Dimension 2, [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
And data, a 15×10 Matrix{VariableRef}:
 L[AJ_Vaccines,1]      L[AJ_Vaccines,2]      …  L[AJ_Vaccines,10]
 L[BB_NCIPD,1]         L[BB_NCIPD,2]            L[BB_NCIPD,10]
 L[China_National,1]   L[China_National,2]      L[China_National,10]
 L[Bharat_Biotech,1]   L[Bharat_Biotech,2]      L[Bharat_Biotech,10]
 L[Bilthoven,1]        L[Bilthoven,2]           L[Bilthoven,10]
 L[Biological_E,1]     L[Biological_E,2]     …  L[Biological_E,10]
 L[GSK,1]              L[GSK,2]                 L[GSK,10]
 L[Haffkine_Bio,1]     L[Haffkine_Bio,2]        L[Haffkine_Bio,10]
 L[LG_Chem,1]          L[LG_Chem,2]             L[LG_Chem,10]
 L[Merck_Sharp,1]      L[Merck_Sharp

In [5]:
# FIX First Stage Y Variables
for (p, time) in data["Y"]
    for (t, val) in time
        value = val  # Directly use the value from the loop
        new_t = parse(Int, t)  # Convert `t` to an integer once
        fix(Y[p, new_t], value)  # Fix the variable
    end
end

In [6]:
# FIX First Stage Q Variables
# Pre-extract Q data once
Q_data = get(data, "Q", Dict())

# Fix First Stage Q Variables
for v in V
    v_data = get(Q_data, v, Dict())  # Access v-level data once
    for p in P_v[v]
        p_data = get(v_data, p, Dict())  # Access p-level data once
        for (t, tau) in F_time_set
            value = get(get(p_data, string(t), Dict()), string(tau), 0)
            fix(Q[v, p, (t, tau)], value; force=true)
        end
    end
end

In [7]:
# FIX First Stage W Variables
W_data = get(data, "W", Dict())  # Extract "W" data once

for p in P
    p_data = get(W_data, p, Dict())  # Access p-level data once
    for (t, tau) in F_time_set
        value = get(get(p_data, string(t), Dict()), string(tau), 0)
        fix(W[p, (t, tau)], value)
    end
end



In [8]:
# FIX First Stage F Variables
F_data = get(data, "F", Dict())  # Extract "F" data once

for a in A
    a_data = get(F_data, a, Dict())  # Access a-level data once
    for (t, tau) in F_time_set
        value = get(get(a_data, string(t), Dict()), string(tau), 0)
        fix(F[a, (t, tau)], value)
    end
end


In [11]:
# FIX First Stage L Variables
F_data = get(data, "L", Dict())  # Extract "L" data once
for p in P
    p_data = get(F_data, p, Dict())  # Access a-level data once
    for t in T
        value = get(p_data, string(t), 0)
        fix(L[p, t], value; force=true)
    end
end

In [12]:
write_to_file(model, "model_test_fix.lp")